<a href="https://colab.research.google.com/github/SVz54/9517/blob/good-cam/9517.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:

# --- Kaggle download (same as before) ---
import os, json, shutil, zipfile, glob, pathlib

os.makedirs('/root/.kaggle', exist_ok=True)
if not os.path.exists('/root/.kaggle/kaggle.json') and os.path.exists('/content/kaggle.json'):
    shutil.move('/content/kaggle.json', '/root/.kaggle/kaggle.json')
!chmod 600 /root/.kaggle/kaggle.json

!pip -q install kaggle
DATA_DIR = "/content/AgroPest12"
os.makedirs(DATA_DIR, exist_ok=True)
!kaggle datasets download -d rupankarmajumdar/crop-pests-dataset -p $DATA_DIR -q

# Unzip (overwrite if re-running)
zip_files = glob.glob(f"{DATA_DIR}/*.zip")
assert zip_files, "Zip not found – did the Kaggle download succeed?"
zip_path = zip_files[0]
!unzip -q -o "$zip_path" -d "$DATA_DIR"

# --- Auto-detect BASE: the folder that contains train/valid/test with images+labels ---
def find_yolo_base(root):
    for p, d, f in os.walk(root):
        if (os.path.isdir(os.path.join(p, "train", "images")) and
            os.path.isdir(os.path.join(p, "train", "labels")) and
            os.path.isdir(os.path.join(p, "valid", "images")) and
            os.path.isdir(os.path.join(p, "valid", "labels"))):
            return p
    return None

BASE = find_yolo_base(DATA_DIR)
assert BASE is not None, f"Could not find YOLO folders under {DATA_DIR}. Found: {os.listdir(DATA_DIR)}"
print("BASE:", BASE)
print("train samples:", len(glob.glob(os.path.join(BASE, "train/images/*.jpg"))))
print("val samples:", len(glob.glob(os.path.join(BASE, "valid/images/*.jpg"))))
print("test samples:", len(glob.glob(os.path.join(BASE, "test/images/*.jpg"))))
BASE = pathlib.Path(BASE)  # keep as Path for later cells

Dataset URL: https://www.kaggle.com/datasets/rupankarmajumdar/crop-pests-dataset
License(s): MIT
BASE: /content/AgroPest12
train samples: 11502
val samples: 1095
test samples: 546


In [6]:
!pip -q install --upgrade ultralytics==8.3.20 opencv-python-headless==4.10.0.84 matplotlib==3.9.2


In [7]:
from pathlib import Path
yaml_path = BASE / "data.yaml"
txt = f"""# AgroPest-12
path: {BASE.as_posix()}
train: train/images
val: valid/images
test: test/images
names:
  0: aphid
  1: armyworm
  2: beetle
  3: bollworm
  4: grasshopper
  5: leafhopper
  6: locust
  7: mealybug
  8: mosquito
  9: moth
  10: sawfly
  11: weevil
"""
yaml_path.write_text(txt)
print("Wrote:", yaml_path)


Wrote: /content/AgroPest12/data.yaml


In [4]:
from ultralytics import YOLO
import os, glob

model = YOLO('yolov8n.pt')   # tiny + fast; swap to yolov8s.pt later if you want

# use a handful of val images for a demo run
val_imgs = sorted(glob.glob(str(BASE / 'valid/images/*.jpg')))[:12]
print("Demo images:", len(val_imgs))

pred_root = "/content/preds"
res = model.predict(val_imgs, conf=0.25, save=True, project=pred_root, name="yolo_preds", exist_ok=True, imgsz=640)
print("Saved predicted images to:", pred_root + "/yolo_preds")


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


100%|██████████| 6.25M/6.25M [00:00<00:00, 122MB/s]


Demo images: 12

0: 640x640 (no detections), 6.9ms
1: 640x640 (no detections), 6.9ms
2: 640x640 1 bear, 6.9ms
3: 640x640 (no detections), 6.9ms
4: 640x640 1 cat, 6.9ms
5: 640x640 1 horse, 6.9ms
6: 640x640 (no detections), 6.9ms
7: 640x640 1 person, 6.9ms
8: 640x640 1 bird, 1 horse, 6.9ms
9: 640x640 1 teddy bear, 6.9ms
10: 640x640 1 bird, 6.9ms
11: 640x640 (no detections), 6.9ms
Speed: 3.9ms preprocess, 6.9ms inference, 26.7ms postprocess per image at shape (1, 3, 640, 640)
Results saved to /content/preds/yolo_preds
Saved predicted images to: /content/preds/yolo_preds


In [ ]:
# Draw a few GT boxes to confirm labels align with images
import glob, os, cv2
from pathlib import Path

samples = sorted(glob.glob(str(BASE / "valid/images/*.jpg")))[:6]
out_dir = "/content/gt_preview"; os.makedirs(out_dir, exist_ok=True)

def read_yolo_labels(lbl_path, W, H):
    boxes = []
    if not os.path.exists(lbl_path): return boxes
    for line in open(lbl_path).read().strip().splitlines():
        c, x, y, w, h = map(float, line.split())
        x1 = int((x - w/2) * W); y1 = int((y - h/2) * H)
        x2 = int((x + w/2) * W); y2 = int((y + h/2) * H)
        boxes.append((int(c), x1, y1, x2, y2))
    return boxes

for p in samples:
    img = cv2.imread(p); H, W = img.shape[:2]
    lbl = str(Path(p).with_suffix("").as_posix().replace("/images/","/labels/") + ".txt")
    for c, x1, y1, x2, y2 in read_yolo_labels(lbl, W, H):
        cv2.rectangle(img, (x1,y1), (x2,y2), (0,255,0), 2)
        cv2.putText(img, str(c), (x1, max(0,y1-5)), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0,255,0), 2)
    cv2.imwrite(os.path.join(out_dir, os.path.basename(p)), img)
print("Wrote previews to", out_dir)


Wrote previews to /content/gt_preview


In [ ]:
# Quick warm-start training on AgroPest-12 (wandb off, clean project name)
import os, glob, torch
os.environ["WANDB_DISABLED"] = "true"
os.environ["WANDB_MODE"] = "disabled"

from ultralytics import YOLO

data_yaml = str(yaml_path)   # from your Cell 3
model = YOLO('yolov8n.pt')

results = model.train(
    data=data_yaml,
    epochs=5,                 # try 10–15 later for better boxes
    imgsz=640,
    batch=16,
    workers=2,
    lr0=0.01,
    freeze=10,
    amp=True,
    device=0 if torch.cuda.is_available() else 'cpu',
    project="agropest_runs",  # <-- no slash
    name="yolov8n_quick",
    exist_ok=True,
)

best = glob.glob("agropest_runs/yolov8n_quick/weights/best.pt")
assert best, "best.pt not found under agropest_runs/yolov8n_quick/weights/"
BEST_WEIGHTS = best[0]
print("BEST_WEIGHTS =", BEST_WEIGHTS)


New https://pypi.org/project/ultralytics/8.3.227 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.20 🚀 Python-3.12.12 torch-2.8.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: task=detect, mode=train, model=yolov8n.pt, data=/content/AgroPest12/data.yaml, epochs=5, time=None, patience=100, batch=16, imgsz=640, save=True, save_period=-1, cache=False, device=0, workers=2, project=agropest_runs, name=yolov8n_quick, exist_ok=True, pretrained=True, optimizer=auto, verbose=True, seed=0, deterministic=True, single_cls=False, rect=False, cos_lr=False, close_mosaic=10, resume=False, amp=True, fraction=1.0, profile=False, freeze=10, multi_scale=False, overlap_mask=True, mask_ratio=4, dropout=0.0, val=True, split=val, save_json=False, save_hybrid=False, conf=None, iou=0.7, max_det=300, half=False, dnn=False, plots=True, source=None, vid_stride=1, stream_buffer=False, visualize=False, augment=False, agnostic_nms=False, classes=None, retina_masks=False, embed=None, show=False

100%|██████████| 755k/755k [00:00<00:00, 28.4MB/s]


Overriding model.yaml nc=80 with nc=12

                   from  n    params  module                                       arguments                     
  0                  -1  1       464  ultralytics.nn.modules.conv.Conv             [3, 16, 3, 2]                 
  1                  -1  1      4672  ultralytics.nn.modules.conv.Conv             [16, 32, 3, 2]                
  2                  -1  1      7360  ultralytics.nn.modules.block.C2f             [32, 32, 1, True]             
  3                  -1  1     18560  ultralytics.nn.modules.conv.Conv             [32, 64, 3, 2]                
  4                  -1  2     49664  ultralytics.nn.modules.block.C2f             [64, 64, 2, True]             
  5                  -1  1     73984  ultralytics.nn.modules.conv.Conv             [64, 128, 3, 2]               
  6                  -1  2    197632  ultralytics.nn.modules.block.C2f             [128, 128, 2, True]           
  7                  -1  1    295424  ultralytic

100%|██████████| 5.35M/5.35M [00:00<00:00, 111MB/s]


AMP: checks passed ✅


...

        5/5      1.36G      1.434      2.148      1.735         47        640: 100%|██████████| 719/719 [03:00<00:00,  3.98it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 35/35 [00:08<00:00,  4.21it/s]

                   all       1095       1341      0.546      0.527      0.539      0.292



5 epochs completed in 0.270 hours.
Optimizer stripped from agropest_runs/yolov8n_quick/weights/last.pt, 6.2MB
Optimizer stripped from agropest_runs/yolov8n_quick/weights/best.pt, 6.2MB

Validating agropest_runs/yolov8n_quick/weights/best.pt...
Ultralytics 8.3.20 🚀 Python-3.12.12 torch-2.8.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
Model summary (fused): 168 layers, 3,007,988 parameters, 0 gradients, 8.1 GFLOPs


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 35/35 [00:10<00:00,  3.26it/s]


                   all       1095       1341      0.545      0.528       0.54      0.292
                 aphid         96        178      0.461      0.528      0.464       0.18
              armyworm         99        110      0.575      0.555      0.547       0.25
                beetle         89        100       0.42       0.31       0.35      0.166
              bollworm         77        139      0.313      0.245      0.199     0.0883
           grasshopper         53         72      0.421      0.236      0.226        0.1
            leafhopper         91        104      0.513      0.529      0.512      0.236
                locust         98        102      0.454      0.471      0.508      0.252
              mealybug        100        101       0.81      0.802      0.898      0.581
              mosquito         77         91      0.605      0.209      0.356      0.191
                  moth         99        107      0.549      0.822      0.768      0.471
                sawfl

In [8]:
from ultralytics import YOLO
import glob, os

BEST_WEIGHTS  # should be set by previous cell
model_trained = YOLO(BEST_WEIGHTS)

val_imgs = sorted(glob.glob(str(BASE / 'valid/images/*.jpg')))[:12]
pred_root = "/content/preds"
res = model_trained.predict(
    val_imgs, conf=0.25, save=True,
    project=pred_root, name="yolo_trained", exist_ok=True, imgsz=640
)
print("Saved predicted images to:", pred_root + "/yolo_trained")


NameError: name 'BEST_WEIGHTS' is not defined

In [ ]:
from google.colab import drive
from ultralytics import YOLO
import os

# --- 1️⃣ Mount Google Drive (do this once) ---
drive.mount('/content/drive')

# --- 2️⃣ Define paths ---
save_dir = "/content/agropest_runs/yolov8s_full"
backup_dir = "/content/drive/MyDrive/agropest_runs/yolov8s_full"

# --- 3️⃣ Train model (same as before) ---
model = YOLO('yolov8s.pt')  # small model
model.train(
    data='/content/AgroPest12/data.yaml',
    epochs=50,        # safe overnight run
    imgsz=640,
    batch=8,
    workers=2,
    patience=10,
    project='agropest_runs',
    name='yolov8s_full',
    cache=True
)

# --- 4️⃣ Auto-copy to Drive right after training ---
os.system(f"mkdir -p '{os.path.dirname(backup_dir)}'")
os.system(f"cp -r '{save_dir}' '{backup_dir}'")
print(f"✅ Training finished. Backup saved to: {backup_dir}")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
New https://pypi.org/project/ultralytics/8.3.227 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.20 🚀 Python-3.12.12 torch-2.8.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: task=detect, mode=train, model=yolov8s.pt, data=/content/AgroPest12/data.yaml, epochs=50, time=None, patience=10, batch=8, imgsz=640, save=True, save_period=-1, cache=True, device=None, workers=2, project=agropest_runs, name=yolov8s_full2, exist_ok=False, pretrained=True, optimizer=auto, verbose=True, seed=0, deterministic=True, single_cls=False, rect=False, cos_lr=False, close_mosaic=10, resume=False, amp=True, fraction=1.0, profile=False, freeze=None, multi_scale=False, overlap_mask=True, mask_ratio=4, dropout=0.0, val=True, split=val, save_json=False, save_hybrid=False, conf=None, iou=0.7, max_det=300, half=False, dnn=False, plots=True, source=None, vid_stride

train: Scanning /content/AgroPest12/train/labels.cache... 11502 images, 3 backgrounds, 0 corrupt: 100%|██████████| 11502/11502 [00:00<?, ?it/s]

train: 19.7GB RAM required to cache images with 50% safety margin but only 6.6/12.7GB available, not caching images ⚠️
albumentations: Blur(p=0.01, blur_limit=(3, 7)), MedianBlur(p=0.01, blur_limit=(3, 7)), ToGray(p=0.01, method='weighted_average', num_output_channels=3), CLAHE(p=0.01, clip_limit=(1.0, 4.0), tile_grid_size=(8, 8))



/usr/local/lib/python3.12/dist-packages/ultralytics/data/augment.py:1850: UserWarning: Argument(s) 'quality_lower' are not valid for transform ImageCompression
  A.ImageCompression(quality_lower=75, p=0.0),
val: Scanning /content/AgroPest12/valid/labels.cache... 1095 images, 0 backgrounds, 0 corrupt: 100%|██████████| 1095/1095 [00:00<?, ?it/s]


WARNING ⚠️ cache='ram' may produce non-deterministic training results. Consider cache='disk' as a deterministic alternative if your disk space allows.


...

      41/50      2.45G     0.9604     0.5494      1.466         25        640: 100%|██████████| 1438/1438 [04:10<00:00,  5.75it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 69/69 [00:09<00:00,  7.08it/s]

                   all       1095       1341       0.87      0.723      0.779      0.454
EarlyStopping: Training stopped early as no improvement observed in last 10 epochs. Best results observed at epoch 31, best model saved as best.pt.
To update EarlyStopping(patience=10) pass a new patience value, i.e. `patience=300` or use `patience=0` to disable EarlyStopping.



41 epochs completed in 3.080 hours.
Optimizer stripped from agropest_runs/yolov8s_full2/weights/last.pt, 22.5MB
Optimizer stripped from agropest_runs/yolov8s_full2/weights/best.pt, 22.5MB

Validating agropest_runs/yolov8s_full2/weights/best.pt...
Ultralytics 8.3.20 🚀 Python-3.12.12 torch-2.8.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
Model summary (fused): 168 layers, 11,130,228 parameters, 0 gradients, 28.5 GFLOPs


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 69/69 [00:10<00:00,  6.33it/s]


                   all       1095       1341      0.844      0.727      0.791      0.467
                 aphid         96        178      0.837      0.596      0.703      0.267
              armyworm         99        110      0.899      0.811      0.877      0.388
                beetle         89        100       0.67       0.63      0.681      0.321
              bollworm         77        139      0.762       0.46      0.511      0.255
           grasshopper         53         72      0.745      0.446      0.509      0.279
            leafhopper         91        104      0.829      0.712      0.808      0.459
                locust         98        102      0.813      0.768      0.822      0.447
              mealybug        100        101       0.96      0.942      0.987      0.758
              mosquito         77         91      0.758      0.637      0.744      0.463
                  moth         99        107      0.961      0.925      0.975      0.684
                sawfl

In [ ]:
from ultralytics import YOLO
import glob, os

# 1️⃣ Mount Drive to access your trained weights
from google.colab import drive
drive.mount('/content/drive')

# 2️⃣ Path to your best trained model
BEST_WEIGHTS = "/content/drive/MyDrive/agropest_runs/yolov8s_full/weights/best.pt"

# 3️⃣ Load model
model_trained = YOLO(BEST_WEIGHTS)

# 4️⃣ Path to your dataset (make sure it's already in Colab)
BASE = "/content/AgroPest12"

# 5️⃣ Get first 10 validation images
val_imgs = sorted(glob.glob(os.path.join(BASE, "valid/images/*.jpg")))[:10]

# 6️⃣ Run prediction on those
pred_root = "/content/preds"
res = model_trained.predict(
    val_imgs,
    conf=0.25,
    save=True,
    project=pred_root,
    name="yolo_trained",
    exist_ok=True,
    imgsz=640
)

print("✅ Saved predicted images to:", os.path.join(pred_root, "yolo_trained"))


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

0: 640x640 (no detections), 800.3ms
1: 640x640 1 weevil, 800.3ms
2: 640x640 1 weevil, 800.3ms
3: 640x640 1 weevil, 800.3ms
4: 640x640 1 weevil, 800.3ms
5: 640x640 1 weevil, 800.3ms
6: 640x640 1 weevil, 800.3ms
7: 640x640 1 weevil, 800.3ms
8: 640x640 2 weevils, 800.3ms
9: 640x640 1 weevil, 800.3ms
Speed: 3.8ms preprocess, 800.3ms inference, 0.5ms postprocess per image at shape (1, 3, 640, 640)
Results saved to /content/preds/yolo_trained
✅ Saved predicted images to: /content/preds/yolo_trained


In [9]:
!pip -q uninstall -y pytorch-grad-cam grad-cam || true
!pip -q install --no-cache-dir "git+https://github.com/jacobgil/pytorch-grad-cam.git"


  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [ ]:
import pytorch_grad_cam, inspect
from pytorch_grad_cam.utils import model_targets
print("grad-cam version:", getattr(pytorch_grad_cam, "__version__", "git"))
print("Has YOLOv8Target:", hasattr(model_targets, "YOLOv8Target"))


grad-cam version: git
Has YOLOv8Target: False


In [20]:
# --- Grad-CAM on one image (class-specific, box-masked, sparse) ---
import os, glob, cv2, torch, numpy as np, torch.nn as nn
from ultralytics import YOLO
from pytorch_grad_cam import GradCAM

# ---- paths ----
BEST_WEIGHTS = "/content/drive/MyDrive/agropest_runs/yolov8s_full/weights/best.pt"
PRED_IMG     = "/content/preds/yolo_trained/Weevil-108-_jpg.rf.8ef561a00c3f440500b751d848d393a1.jpg"
DATA_ROOT    = "/content/AgroPest12"   # where your dataset is mounted in Colab

# Try to use the original clean image (not the drawn prediction)
base = os.path.basename(PRED_IMG)
cand1 = os.path.join(DATA_ROOT, "valid/images", base)
IMG_PATH = cand1 if os.path.exists(cand1) else PRED_IMG
if IMG_PATH is PRED_IMG:
    print("Note: original image not found; using the drawn prediction image.")

# ---- CAM/overlay params (tuned to avoid purple slab) ----
INPUT_SZ = 640
LO_PCT, HI_PCT = 85.0, 99.0    # stronger clipping => higher contrast
TOPK = 0.12                    # paint only top ~12% hottest pixels
HEAT_W, IMG_W = 0.45, 0.55     # lighter heat, more original image

# ---------------- helpers ----------------
def pct_norm(x, lo=75.0, hi=99.3, eps=1e-6):
    lo_v = np.percentile(x, lo); hi_v = np.percentile(x, hi)
    if hi_v - lo_v < 1e-8:
        lo_v, hi_v = x.min(), x.max() + eps
    return np.clip((x - lo_v) / (hi_v - lo_v + eps), 0, 1)

def penultimate_conv(torch_model):
    convs = [m for m in torch_model.modules() if isinstance(m, nn.Conv2d)]
    if len(convs) < 2:
        raise RuntimeError("Need ≥2 convs for penultimate.")
    return convs[-2]  # penultimate Conv2d often gives crisper CAMs

def imread_rgb(p):
    bgr = cv2.imread(p, cv2.IMREAD_COLOR)
    if bgr is None:
        raise FileNotFoundError(p)
    return cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)

def imwrite_rgb(p, rgb):
    cv2.imwrite(p, cv2.cvtColor(rgb, cv2.COLOR_RGB2BGR))

# ---------------- models ----------------
device = "cuda" if torch.cuda.is_available() else "cpu"
model_det = YOLO(BEST_WEIGHTS)     # detector (to get box + class)
model_cam = YOLO(BEST_WEIGHTS)     # clean graph for hooks
model_cam.model.to(device).eval()
target_layers = [penultimate_conv(model_cam.model)]

# ---------------- detect on full image ----------------
rgb = imread_rgb(IMG_PATH)
H, W = rgb.shape[:2]

def detect_with_fallback(img):
    r = model_det.predict(source=img, conf=0.20, imgsz=640, verbose=False)[0]
    if r.boxes is None or len(r.boxes) == 0:
        r = model_det.predict(source=img, conf=0.10, imgsz=800, verbose=False)[0]
    return r

res = detect_with_fallback(rgb)
assert res.boxes is not None and len(res.boxes) > 0, "No detections even after fallback."

# take highest-confidence detection
i = int(torch.argmax(res.boxes.conf).item())
x1, y1, x2, y2 = map(int, res.boxes.xyxy[i].tolist())
x1, y1 = max(0, x1), max(0, y1)
x2, y2 = min(W - 1, x2), min(H - 1, y2)
det_conf = float(res.boxes.conf[i].item())
det_cls = int(res.boxes.cls[i].item())
CLASS_NAMES = model_det.model.names if hasattr(model_det.model, "names") else model_det.names
print(f"[det] box=({x1},{y1},{x2},{y2}) conf={det_conf:.2f} cls={det_cls} ({CLASS_NAMES[det_cls]})")

# ---------------- crop & prep CAM input ----------------
crop = rgb[y1:y2, x1:x2]
crop_resz = cv2.resize(crop, (INPUT_SZ, INPUT_SZ), interpolation=cv2.INTER_LINEAR)
inp = torch.from_numpy((crop_resz.astype(np.float32) / 255.).transpose(2, 0, 1)).unsqueeze(0).to(device)
inp.requires_grad_(True)

# ---------------- pick a class target from the CROP ----------------
r_crop = model_det.predict(source=crop, conf=0.01, imgsz=INPUT_SZ, verbose=False)[0]
if r_crop.boxes is not None and len(r_crop.boxes) > 0:
    j = int(torch.argmax(r_crop.boxes.conf).item())
    top_cls = int(r_crop.boxes.cls[j].item())
    print(f"[crop] cls={top_cls} ({CLASS_NAMES[top_cls]}), conf={float(r_crop.boxes.conf[j].item()):.2f}")
else:
    top_cls = det_cls
    print(f"[crop] no boxes; falling back to det cls={top_cls} ({CLASS_NAMES[top_cls]})")

class YoloRowTarget:
    def __init__(self, cls_idx, model_ref):
        self.cls_idx = int(cls_idx)
        self.nc = model_ref.model.model[-1].nc
    def __call__(self, outputs):
        pred = outputs[0]  # (num_preds, 4+nc) or (num_preds, 5+nc)
        if pred.size(1) > 4 + self.nc:
            obj = pred[:, 4].sigmoid()
            cls = pred[:, 5 + self.cls_idx].sigmoid()
            return (obj * cls).max()
        else:
            cls = pred[:, 4 + self.cls_idx].sigmoid()
            return cls.max()

target = YoloRowTarget(top_cls, model_cam)

# ---------------- Grad-CAM ----------------
model_cam.model.zero_grad(set_to_none=True)
gcam = GradCAM(model=model_cam.model, target_layers=target_layers)
cam_map = gcam(input_tensor=inp, targets=[target])[0]
gcam.activations_and_grads.release()
del gcam

# ---------------- overlay (sparse + gentle) ----------------
thr = np.quantile(cam_map, 1 - TOPK)
mask = (cam_map >= thr).astype(np.float32)

cam_map = pct_norm(cam_map, LO_PCT, HI_PCT)
heat = cv2.applyColorMap((cam_map * 255).astype(np.uint8), cv2.COLORMAP_JET)
heat = cv2.cvtColor(heat, cv2.COLOR_BGR2RGB).astype(np.float32)
heat = heat * mask[..., None]  # sparsify

blend = np.clip(HEAT_W * heat + IMG_W * crop_resz.astype(np.float32), 0, 255).astype(np.uint8)

# paste back
out = rgb.copy()
out[y1:y2, x1:x2] = cv2.resize(blend, (x2 - x1, y2 - y1), interpolation=cv2.INTER_LINEAR)

os.makedirs("/content/gradcam", exist_ok=True)
save_path = f"/content/gradcam/{os.path.basename(IMG_PATH).replace('.jpg','_gradcam.jpg')}"
imwrite_rgb(save_path, out)
print("Saved:", save_path)


[det] box=(168,197,470,458) conf=0.48 cls=11 (weevil)
[crop] cls=2 (beetle), conf=0.12
Saved: /content/gradcam/Weevil-108-_jpg.rf.8ef561a00c3f440500b751d848d393a1_gradcam.jpg


In [21]:
# Batch Grad-CAM overlay + simple sanity metrics (reuses your Cell-13 approach)
import os, glob, cv2, torch, numpy as np, torch.nn as nn
from ultralytics import YOLO
from pytorch_grad_cam import GradCAM

BEST_WEIGHTS = "/content/drive/MyDrive/agropest_runs/yolov8s_full/weights/best.pt"
DATA_ROOT = "/content/AgroPest12"
OUT_DIR = "/content/gradcam_batch"; os.makedirs(OUT_DIR, exist_ok=True)

INPUT_SZ = 640
LO_PCT, HI_PCT = 85.0, 99.0
TOPK = 0.12
HEAT_W, IMG_W = 0.45, 0.55

def pct_norm(x, lo=75.0, hi=99.3, eps=1e-6):
    lo_v = np.percentile(x, lo); hi_v = np.percentile(x, hi)
    if hi_v - lo_v < 1e-8: lo_v, hi_v = x.min(), x.max() + eps
    return np.clip((x - lo_v) / (hi_v - lo_v + eps), 0, 1)

def penultimate_conv(torch_model):
    convs = [m for m in torch_model.modules() if isinstance(m, nn.Conv2d)]
    if len(convs) < 2: raise RuntimeError("Need >=2 convs")
    return convs[-2]

def imread_rgb(p):
    bgr = cv2.imread(p, cv2.IMREAD_COLOR)
    if bgr is None: raise FileNotFoundError(p)
    return cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)

def imwrite_rgb(p, rgb):
    cv2.imwrite(p, cv2.cvtColor(rgb, cv2.COLOR_RGB2BGR))

device = "cuda" if torch.cuda.is_available() else "cpu"
det = YOLO(BEST_WEIGHTS)
cam_wrap = YOLO(BEST_WEIGHTS)
cam_wrap.model.to(device).eval()
target_layers = [penultimate_conv(cam_wrap.model)]

val_imgs = sorted(glob.glob(os.path.join(DATA_ROOT, "valid/images/*.jpg")))[:20]

def gradcam_one(img_path):
    rgb = imread_rgb(img_path); H,W = rgb.shape[:2]
    r = det.predict(source=rgb, conf=0.20, imgsz=INPUT_SZ, verbose=False)[0]
    if r.boxes is None or len(r.boxes) == 0:
        r = det.predict(source=rgb, conf=0.10, imgsz=800, verbose=False)[0]
    if r.boxes is None or len(r.boxes) == 0:
        return None

    i = int(torch.argmax(r.boxes.conf).item())
    x1,y1,x2,y2 = map(int, r.boxes.xyxy[i].tolist())
    x1,y1 = max(0,x1), max(0,y1); x2,y2 = min(W-1,x2), min(H-1,y2)
    det_cls = int(r.boxes.cls[i].item())

    crop = rgb[y1:y2, x1:x2]
    crop_resz = cv2.resize(crop, (INPUT_SZ, INPUT_SZ), interpolation=cv2.INTER_LINEAR)
    inp = torch.from_numpy((crop_resz.astype(np.float32)/255.).transpose(2,0,1)).unsqueeze(0).to(device)
    inp.requires_grad_(True)

    # pick class from crop if present
    r_crop = det.predict(source=crop, conf=0.01, imgsz=INPUT_SZ, verbose=False)[0]
    if r_crop.boxes is not None and len(r_crop.boxes) > 0:
        j = int(torch.argmax(r_crop.boxes.conf).item()); top_cls = int(r_crop.boxes.cls[j].item())
    else:
        top_cls = det_cls

    class YoloRowTarget:
        def __init__(self, cls_idx, model_ref):
            self.cls_idx = int(cls_idx); self.nc = model_ref.model.model[-1].nc
        def __call__(self, outputs):
            pred = outputs[0]
            if pred.size(1) > 4 + self.nc:
                obj = pred[:,4].sigmoid(); cls = pred[:,5 + self.cls_idx].sigmoid()
                return (obj*cls).max()
            else:
                cls = pred[:,4 + self.cls_idx].sigmoid()
                return cls.max()

    target = YoloRowTarget(top_cls, cam_wrap)

    gcam = GradCAM(model=cam_wrap.model, target_layers=target_layers)
    cam_map = gcam(input_tensor=inp, targets=[target])[0]
    gcam.activations_and_grads.release(); del gcam

    thr = np.quantile(cam_map, 1 - TOPK)
    mask_sparse = (cam_map >= thr).astype(np.float32)

    cam_n = pct_norm(cam_map, LO_PCT, HI_PCT)
    heat = cv2.applyColorMap((cam_n * 255).astype(np.uint8), cv2.COLORMAP_JET)
    heat = cv2.cvtColor(heat, cv2.COLOR_BGR2RGB).astype(np.float32)
    heat = heat * mask_sparse[..., None]

    blend = np.clip(HEAT_W*heat + IMG_W*crop_resz.astype(np.float32), 0, 255).astype(np.uint8)
    out = rgb.copy()
    out[y1:y2, x1:x2] = cv2.resize(blend, (x2-x1, y2-y1), interpolation=cv2.INTER_LINEAR)

    # simple metrics
    # resize sparse mask back to bbox, then into full image canvas
    sparse_full = np.zeros((H,W), dtype=np.float32)
    sparse_roi = cv2.resize(mask_sparse, (x2-x1, y2-y1), interpolation=cv2.INTER_NEAREST)
    sparse_full[y1:y2, x1:x2] = sparse_roi
    # inside-bbox activation ratio
    bbox_area = max(1, (x2-x1)*(y2-y1))
    inside = sparse_roi.sum() / bbox_area
    # sparsity (fraction of active pixels in full image)
    sparsity = sparse_full.mean()

    return out, inside, sparsity

rows = []
for p in val_imgs:
    res = gradcam_one(p)
    if res is None:
        print(f"SKIP (no detections): {os.path.basename(p)}")
        continue
    out, inside, sparsity = res
    save_p = os.path.join(OUT_DIR, os.path.basename(p).replace(".jpg","_gradcam.jpg"))
    imwrite_rgb(save_p, out)
    rows.append((os.path.basename(p), round(inside,4), round(sparsity,4)))
    print(f"Saved {save_p} | inside_bbox={inside:.3f} | sparsity={sparsity:.3f}")

print("\nSummary (filename, inside_bbox, sparsity):")
for r in rows:
    print(r)


Saved /content/gradcam_batch/Weevil-101-_jpg.rf.7b2714887709397bf4467aee16ea2e79_gradcam.jpg | inside_bbox=1.000 | sparsity=0.575
Saved /content/gradcam_batch/Weevil-102-_jpg.rf.34c74f7621cdea247c335cac46f5c73f_gradcam.jpg | inside_bbox=1.000 | sparsity=0.964
Saved /content/gradcam_batch/Weevil-108-_jpg.rf.8ef561a00c3f440500b751d848d393a1_gradcam.jpg | inside_bbox=1.000 | sparsity=0.192
Saved /content/gradcam_batch/Weevil-109-_jpg.rf.1846449af8616a92737824c761183fb2_gradcam.jpg | inside_bbox=1.000 | sparsity=0.641
Saved /content/gradcam_batch/Weevil-116-_jpg.rf.76f91165aa3a1d41954ef906c6483b43_gradcam.jpg | inside_bbox=1.000 | sparsity=0.482
Saved /content/gradcam_batch/Weevil-119-_jpg.rf.fb960fd8effbb970015d32c7b0dcb052_gradcam.jpg | inside_bbox=1.000 | sparsity=0.573
Saved /content/gradcam_batch/Weevil-128-_jpg.rf.a60bfb8b73412b8592a8a96c6b63d812_gradcam.jpg | inside_bbox=1.000 | sparsity=0.587
Saved /content/gradcam_batch/Weevil-132-_jpg.rf.cd55c6ee48ce4a4f0e6a3226dce427eb_gradcam.j

In [22]:
from google.colab import drive
drive.mount('/content/drive')

!cp -r /content/gradcam_batch /content/drive/MyDrive/gradcam_batch_backup


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
